In [1]:
%pip install faker

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 2.3 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 2.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.5 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import random
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)

# Departments
departments = {
    "Operations": 1,
    "Customs": 2,
    "Finance": 3,
    "HR": 4,
    "IT": 5
}

# Roles
roles = {
    "employee": 1,
    "manager": 2,
    "ceo": 3
}

employees = []

# 1 CEO
employees.append({
    "first_name": fake.first_name(),
    "last_name": fake.last_name(),
    "email": "ceo@nilegate.com",
    "role_id": roles["ceo"],
    "department_id": departments["Operations"],
    "hire_date": fake.date_between(start_date="-15y", end_date="-5y"),
    "status": "Active"
})

# 2 managers per department
manager_counter = 1

for department_name, department_id in departments.items():
    for _ in range(2):
        first_name = fake.first_name()
        last_name = fake.last_name()

        employees.append({
            "first_name": first_name,
            "last_name": last_name,
            "email": f"manager{manager_counter}@nilegate.com",
            "role_id": roles["manager"],
            "department_id": department_id,
            "hire_date": fake.date_between(start_date="-12y", end_date="-2y"),
            "status": "Active"
        })

        manager_counter += 1

# 89 regular employees
for i in range(1, 90):
    first_name = fake.first_name()
    last_name = fake.last_name()
    department_id = random.choice(list(departments.values()))

    employees.append({
        "first_name": first_name,
        "last_name": last_name,
        "email": f"employee{i}@nilegate.com",
        "role_id": roles["employee"],
        "department_id": department_id,
        "hire_date": fake.date_between(start_date="-8y", end_date="-30d"),
        "status": random.choices(
            ["Active", "Inactive"],
            weights=[95, 5],
            k=1
        )[0]
    })

employees_df = pd.DataFrame(employees)

print(employees_df.shape)
print("\nRole distribution:")
print(employees_df["role_id"].value_counts().sort_index())

print("\nDepartment distribution:")
print(employees_df["department_id"].value_counts().sort_index())

employees_df.head()

(100, 7)

Role distribution:
role_id
1    89
2    10
3     1
Name: count, dtype: int64

Department distribution:
department_id
1    21
2    19
3    19
4    18
5    23
Name: count, dtype: int64


,first_name,last_name,email,role_id,department_id,hire_date,status
0,Danielle,Johnson,ceo@nilegate.com,3,1,2016-05-17,Active
1,John,Taylor,manager1@nilegate.com,2,1,2016-06-10,Active
2,Erica,Mcclain,manager2@nilegate.com,2,1,2016-03-06,Active
3,Brittany,Johnson,manager3@nilegate.com,2,2,2016-04-17,Active
4,Jeffery,Wagner,manager4@nilegate.com,2,2,2015-02-24,Active


In [3]:
employees_df.to_csv("employees.csv", index=False)

print("Saved:", "employees.csv")
print("Rows:", len(employees_df))

Saved: employees.csv
Rows: 100


In [4]:
import pandas as pd

ports_df = pd.read_csv("ports.csv")

print("Shape:", ports_df.shape)
print("\nColumns:")
print(ports_df.columns.tolist())

ports_df.head()

Shape: (837, 8)

Columns:
['LOCODE', 'Name', 'Country', 'Function', 'Status', 'outflows', 'lon', 'lat']


,LOCODE,Name,Country,Function,Status,outflows,lon,lat
0,AEAUH,Abu Dhabi,United Arab Emirates,1-345---,AI,4.159714e+04,54.366667,24.466667
1,AERUW,Ar Ruways,United Arab Emirates,1-------,RL,1.665560e+05,52.733333,24.116667
2,AEKLF,Khor al Fakkan,United Arab Emirates,1-------,RL,7.904065e+05,56.350000,25.333333
3,AEMKH,Mina Khalid,United Arab Emirates,1-3-----,RL,6.469650e+05,55.366667,25.350000
4,AEKHL,Mina Khalifa/Abu Dhabi,United Arab Emirates,1-------,RL,1.834146e+07,54.666667,24.833333


In [7]:
import pandas as pd
import random
import numpy as np

random.seed(42)
np.random.seed(42)


toll_df = pd.read_csv("toll_rate_bands.csv")
# Canonical vessel categories from the validated toll data
vessel_types = toll_df["vessel_category"].unique().tolist()

# Countries available from the real ports dataset
countries = ports_df["Country"].dropna().unique().tolist()

# Real port LOCODEs
locodes = ports_df["LOCODE"].dropna().unique().tolist()

# Vessel SCNT ranges (approximate operational ranges for synthetic data)
scnt_ranges = {
    "Crude Oil Tankers": (5000, 300000),
    "Petroleum Product Tankers": (3000, 150000),
    "Dry Bulk Vessels": (3000, 200000),
    "Liquefied Petroleum Gas (LPG) Carriers": (2000, 100000),
    "Liquefied Natural Gas (LNG) Carriers": (20000, 180000),
    "Chemical Tankers & Other Liquid Bulk Tankers": (2000, 80000),
    "General Cargo / Multi-Purpose / Heavy Lift Vessels": (1000, 70000),
    "Roll-On/Roll-Off (Ro/Ro) Vessels": (3000, 80000),
    "Vehicles Carriers": (5000, 100000),
    "Cruise Ships": (10000, 150000),
    "Special Floating Units": (500, 100000),
    "Other Vessels": (500, 50000),
    "Containerships": (5000, 200000)
}

vessels = []

for i in range(1, 501):

    vessel_type = random.choice(vessel_types)

    # Generate SCNT within the type-specific range
    min_tons, max_tons = scnt_ranges[vessel_type]
    scnt_tons = round(random.uniform(min_tons, max_tons), 2)

    # Select two different real ports
    origin, destination = random.sample(locodes, 2)

    vessels.append({
        "imo_number": str(9000000 + i),
        "vessel_name": f"{vessel_type.split()[0]} Voyager {i:03d}",
        "vessel_type": vessel_type,
        "flag_country": random.choice(countries),
        "scnt_tons": scnt_tons,
        "origin_locode": origin,
        "destination_locode": destination,
        "condition": random.choice(["Laden", "Ballast"]),
        "status": random.choices(
            ["In Transit", "At Port", "Anchored", "Completed"],
            weights=[45, 25, 15, 15],
            k=1
        )[0]
    })

vessels_df = pd.DataFrame(vessels)

print("Shape:", vessels_df.shape)

print("\nVessel type distribution:")
print(vessels_df["vessel_type"].value_counts())

print("\nCondition distribution:")
print(vessels_df["condition"].value_counts())

print("\nStatus distribution:")
print(vessels_df["status"].value_counts())

vessels_df.head()

Shape: (500, 9)

Vessel type distribution:
vessel_type
Petroleum Product Tankers                             53
Cruise Ships                                          50
Liquefied Petroleum Gas (LPG) Carriers                45
General Cargo / Multi-Purpose / Heavy Lift Vessels    44
Liquefied Natural Gas (LNG) Carriers                  44
Containerships                                        40
Other Vessels                                         38
Special Floating Units                                35
Dry Bulk Vessels                                      33
Vehicles Carriers                                     32
Chemical Tankers & Other Liquid Bulk Tankers          31
Roll-On/Roll-Off (Ro/Ro) Vessels                      29
Crude Oil Tankers                                     26
Name: count, dtype: int64

Condition distribution:
condition
Ballast    252
Laden      248
Name: count, dtype: int64

Status distribution:
status
In Transit    219
At Port       130
Anchored       78
Comp

,imo_number,vessel_name,vessel_type,flag_country,scnt_tons,origin_locode,destination_locode,condition,status
0,9000001,Special Voyager 001,Special Floating Units,"Korea, Republic of",11577.44,PKGWD,MXCOA,Laden,In Transit
1,9000002,Petroleum Voyager 002,Petroleum Product Tankers,Oman,102474.82,ESALC,CRMOB,Ballast,In Transit
2,9000003,Petroleum Voyager 003,Petroleum Product Tankers,Bangladesh,35139.78,BZBZE,IDSOQ,Laden,Anchored
3,9000004,Other Voyager 004,Other Vessels,Mozambique,27474.60,ITCVV,USNTD,Ballast,Anchored
4,9000005,Crude Voyager 005,Crude Oil Tankers,"Taiwan, Province of China",228848.17,GBGRG,MYSDK,Ballast,In Transit


In [8]:
print("Duplicate IMO:", vessels_df["imo_number"].duplicated().sum())

print(
    "Same origin/destination:",
    (vessels_df["origin_locode"] == vessels_df["destination_locode"]).sum()
)

print(
    "Invalid origin LOCODEs:",
    (~vessels_df["origin_locode"].isin(ports_df["LOCODE"])).sum()
)

print(
    "Invalid destination LOCODEs:",
    (~vessels_df["destination_locode"].isin(ports_df["LOCODE"])).sum()
)

print(
    "Invalid vessel types:",
    (~vessels_df["vessel_type"].isin(vessel_types)).sum()
)

print(
    "Missing values:",
    vessels_df.isnull().sum().sum()
)

Duplicate IMO: 0
Same origin/destination: 0
Invalid origin LOCODEs: 0
Invalid destination LOCODEs: 0
Invalid vessel types: 0
Missing values: 0


In [9]:
vessels_df.to_csv("vessels.csv", index=False)

print("Saved: vessels.csv")
print("Rows:", len(vessels_df))

Saved: vessels.csv
Rows: 500


In [10]:
import random
import pandas as pd

random.seed(42)

container_types = ["Dry", "Reefer", "Open Top", "Flat Rack"]

container_statuses = [
    "Empty",
    "Loaded",
    "In Transit",
    "At Port",
    "Delivered"
]

containers = []

for i in range(1, 1001):
    vessel = vessels_df.sample(1).iloc[0]

    origin = vessel["origin_locode"]
    destination = vessel["destination_locode"]

    containers.append({
        "container_number": f"NGCU{i:07d}",
        "container_type": random.choice(container_types),
        "size_ft": random.choice([20, 40]),
        "status": random.choice(container_statuses),
        "vessel_imo": vessel["imo_number"],
        "origin_locode": origin,
        "destination_locode": destination
    })

containers_df = pd.DataFrame(containers)

print("Shape:", containers_df.shape)
print(containers_df.head())

Shape: (1000, 7)
  container_number container_type  size_ft      status vessel_imo  \
0      NGCU0000001            Dry       20  In Transit    9000362   
1      NGCU0000002         Reefer       20      Loaded    9000307   
2      NGCU0000003            Dry       20   Delivered    9000116   
3      NGCU0000004      Flat Rack       20       Empty    9000213   
4      NGCU0000005            Dry       20      Loaded    9000470   

  origin_locode destination_locode  
0         PHCEB              GEPTI  
1         USMOB              NZAKL  
2         ARROS              GDSTG  
3         ESSPC              ROCND  
4         TOTBU              ISGRF  


In [11]:
print("Duplicate container numbers:",
      containers_df["container_number"].duplicated().sum())

print("Invalid vessel IMO:",
      (~containers_df["vessel_imo"].isin(vessels_df["imo_number"])).sum())

print("Same origin/destination:",
      (containers_df["origin_locode"] == containers_df["destination_locode"]).sum())

print("Invalid origin LOCODEs:",
      (~containers_df["origin_locode"].isin(ports_df["LOCODE"])).sum())

print("Invalid destination LOCODEs:",
      (~containers_df["destination_locode"].isin(ports_df["LOCODE"])).sum())

print("Missing values:",
      containers_df.isnull().sum().sum())

Duplicate container numbers: 0
Invalid vessel IMO: 0
Same origin/destination: 0
Invalid origin LOCODEs: 0
Invalid destination LOCODEs: 0
Missing values: 0


In [13]:
# Map vessel IMO to PostgreSQL vessel_id
vessel_id_map = {
    imo: idx
    for idx, imo in enumerate(vessels_df["imo_number"], start=1)
}

containers_df["vessel_id"] = containers_df["vessel_imo"].map(vessel_id_map)

# Remove the temporary IMO column
containers_df = containers_df.drop(columns=["vessel_imo"])

# Reorder columns to match PostgreSQL table
containers_df = containers_df[
    [
        "container_number",
        "container_type",
        "size_ft",
        "status",
        "vessel_id",
        "origin_locode",
        "destination_locode"
    ]
]

print(containers_df.head())
print("\nShape:", containers_df.shape)
print("Missing vessel_id:", containers_df["vessel_id"].isnull().sum())

  container_number container_type  size_ft      status  vessel_id  \
0      NGCU0000001            Dry       20  In Transit        362   
1      NGCU0000002         Reefer       20      Loaded        307   
2      NGCU0000003            Dry       20   Delivered        116   
3      NGCU0000004      Flat Rack       20       Empty        213   
4      NGCU0000005            Dry       20      Loaded        470   

  origin_locode destination_locode  
0         PHCEB              GEPTI  
1         USMOB              NZAKL  
2         ARROS              GDSTG  
3         ESSPC              ROCND  
4         TOTBU              ISGRF  

Shape: (1000, 7)
Missing vessel_id: 0


In [14]:
containers_df.to_csv("containers.csv", index=False)

print("Updated: containers.csv")
print("Rows:", len(containers_df))
print("Columns:", list(containers_df.columns))

Updated: containers.csv
Rows: 1000
Columns: ['container_number', 'container_type', 'size_ft', 'status', 'vessel_id', 'origin_locode', 'destination_locode']


In [15]:
toll_df = pd.read_csv("toll_rate_bands.csv")

print(toll_df.shape)
print(toll_df.columns.tolist())
print(toll_df["vessel_category"].nunique())

(177, 7)
['vessel_category', 'laden_ballast', 'band_order', 'band_label', 'band_min_tons', 'band_max_tons', 'rate_sdr_per_ton']
13


In [16]:
def calculate_suez_toll(vessel_category, condition, scnt_tons, toll_df):
    bands = toll_df[
        (toll_df["vessel_category"] == vessel_category) &
        (toll_df["laden_ballast"] == condition)
    ].sort_values("band_order")

    total = 0.0

    for _, band in bands.iterrows():
        band_min = band["band_min_tons"]
        band_max = band["band_max_tons"]

        if scnt_tons <= band_min:
            continue

        upper_limit = scnt_tons if pd.isna(band_max) else min(scnt_tons, band_max)

        tons_in_band = upper_limit - band_min

        if tons_in_band > 0:
            total += tons_in_band * band["rate_sdr_per_ton"]

    return round(total, 2)

In [17]:
test_vessel = vessels_df.iloc[0]

test_vessel

imo_number                           9000001
vessel_name              Special Voyager 001
vessel_type           Special Floating Units
flag_country              Korea, Republic of
scnt_tons                           11577.44
origin_locode                          PKGWD
destination_locode                     MXCOA
condition                              Laden
status                            In Transit
Name: 0, dtype: object

In [18]:
test_toll = calculate_suez_toll(
    vessel_category=test_vessel["vessel_type"],
    condition=test_vessel["condition"],
    scnt_tons=test_vessel["scnt_tons"],
    toll_df=toll_df
)

print("Vessel Type:", test_vessel["vessel_type"])
print("Condition:", test_vessel["condition"])
print("SCNT:", test_vessel["scnt_tons"])
print("Calculated Toll (SDR):", test_toll)

Vessel Type: Special Floating Units
Condition: Laden
SCNT: 11577.44
Calculated Toll (SDR): 96395.9


In [19]:
bands_test = toll_df[
    (toll_df["vessel_category"] == test_vessel["vessel_type"]) &
    (toll_df["laden_ballast"] == test_vessel["condition"]) &
    (toll_df["band_min_tons"] < test_vessel["scnt_tons"])
].sort_values("band_order").copy()

bands_test["tons_used"] = (
    bands_test["band_max_tons"]
    .fillna(test_vessel["scnt_tons"])
    .clip(upper=test_vessel["scnt_tons"])
    - bands_test["band_min_tons"]
)

bands_test["amount_sdr"] = (
    bands_test["tons_used"] *
    bands_test["rate_sdr_per_ton"]
)

print(
    bands_test[
        [
            "band_label",
            "band_min_tons",
            "band_max_tons",
            "rate_sdr_per_ton",
            "tons_used",
            "amount_sdr"
        ]
    ]
)

print("\nTotal SDR:", bands_test["amount_sdr"].sum())

          band_label  band_min_tons  band_max_tons  rate_sdr_per_ton  \
140  First 5000 Tons              0         5000.0             10.42   
141   Next 5000 Tons           5000        10000.0              6.90   
142  Next 10000 Tons          10000        20000.0              6.21   

     tons_used  amount_sdr  
140    5000.00  52100.0000  
141    5000.00  34500.0000  
142    1577.44   9795.9024  

Total SDR: 96395.9024


In [20]:
import random
from datetime import date, timedelta

random.seed(42)

currencies = pd.read_csv("accepted_currencies.csv")["currency_code"].tolist()

print("Currencies:", currencies)
print("Number of currencies:", len(currencies))

Currencies: ['USD', 'GBP', 'EUR', 'JPY', 'CAD', 'SEK', 'DKK', 'NOK', 'CHF', 'CNY']
Number of currencies: 10


In [21]:
import random
from datetime import date, timedelta

random.seed(42)

invoices = []

for i in range(1, 1001):
    vessel = vessels_df.sample(1).iloc[0]

    vessel_type = vessel["vessel_type"]
    condition = vessel["condition"]
    scnt_tons = vessel["scnt_tons"]

    # Special Floating Units only has Laden rates
    if vessel_type == "Special Floating Units":
        condition = "Laden"

    amount_sdr = calculate_suez_toll(
        vessel_category=vessel_type,
        condition=condition,
        scnt_tons=scnt_tons,
        toll_df=toll_df
    )

    invoice_date = date(2025, 1, 1) + timedelta(
        days=random.randint(0, 364)
    )

    invoices.append({
        "invoice_number": f"INV-{i:07d}",
        "vessel_imo": vessel["imo_number"],
        "invoice_date": invoice_date,
        "currency": random.choice(currencies),
        "scnt_tons": scnt_tons,
        "amount_sdr": amount_sdr,
        "description": f"Suez Canal transit dues - {vessel_type}",
        "status": random.choice(["Paid", "Pending", "Issued"])
    })

invoices_df = pd.DataFrame(invoices)

print("Shape:", invoices_df.shape)
print(invoices_df.head())

Shape: (1000, 8)
  invoice_number vessel_imo invoice_date currency  scnt_tons  amount_sdr  \
0    INV-0000001    9000017   2025-11-24      GBP   62949.78   300311.68   
1    INV-0000002    9000083   2025-05-21      JPY   51036.72   237291.91   
2    INV-0000003    9000042   2025-03-13      GBP   96933.52   440741.34   
3    INV-0000004    9000370   2025-10-07      GBP   47483.36   242312.60   
4    INV-0000005    9000154   2025-08-05      USD  187150.54   469128.50   

                                         description  status  
0  Suez Canal transit dues - General Cargo / Mult...    Paid  
1        Suez Canal transit dues - Vehicles Carriers    Paid  
2  Suez Canal transit dues - Liquefied Petroleum ...  Issued  
3  Suez Canal transit dues - General Cargo / Mult...  Issued  
4        Suez Canal transit dues - Crude Oil Tankers    Paid  


In [22]:
invoices_df["amount"] = invoices_df["amount_sdr"]

invoices_df = invoices_df.drop(columns=["currency", "amount_sdr"])

print(invoices_df.shape)
print(invoices_df.columns.tolist())
print(invoices_df.head())

(1000, 7)
['invoice_number', 'vessel_imo', 'invoice_date', 'scnt_tons', 'description', 'status', 'amount']
  invoice_number vessel_imo invoice_date  scnt_tons  \
0    INV-0000001    9000017   2025-11-24   62949.78   
1    INV-0000002    9000083   2025-05-21   51036.72   
2    INV-0000003    9000042   2025-03-13   96933.52   
3    INV-0000004    9000370   2025-10-07   47483.36   
4    INV-0000005    9000154   2025-08-05  187150.54   

                                         description  status     amount  
0  Suez Canal transit dues - General Cargo / Mult...    Paid  300311.68  
1        Suez Canal transit dues - Vehicles Carriers    Paid  237291.91  
2  Suez Canal transit dues - Liquefied Petroleum ...  Issued  440741.34  
3  Suez Canal transit dues - General Cargo / Mult...  Issued  242312.60  
4        Suez Canal transit dues - Crude Oil Tankers    Paid  469128.50  


In [23]:
vessel_id_map = {
    imo: idx
    for idx, imo in enumerate(vessels_df["imo_number"], start=1)
}

invoices_df["vessel_id"] = invoices_df["vessel_imo"].map(vessel_id_map)

print("Missing vessel_id:", invoices_df["vessel_id"].isnull().sum())

Missing vessel_id: 0


In [24]:
def get_toll_band_id(vessel_category, condition, scnt_tons, toll_df):
    bands = toll_df[
        (toll_df["vessel_category"] == vessel_category) &
        (toll_df["laden_ballast"] == condition) &
        (toll_df["band_min_tons"] < scnt_tons) &
        (
            toll_df["band_max_tons"].isna() |
            (toll_df["band_max_tons"] >= scnt_tons)
        )
    ].sort_values("band_order")

    if bands.empty:
        return None

    return bands.iloc[0]["band_order"]

In [26]:
print(invoices_df.shape)
print(invoices_df.columns.tolist())

(1000, 8)
['invoice_number', 'vessel_imo', 'invoice_date', 'scnt_tons', 'description', 'status', 'amount', 'vessel_id']


In [28]:
import psycopg

conn = psycopg.connect(
    "host=localhost port=5432 dbname=ports_logistics user=postgres password=postgres"
)

print("Connected!")

Connected!


In [29]:
toll_bands_db = pd.read_sql_query(
    """
    SELECT
        id,
        vessel_category,
        laden_ballast,
        band_order,
        band_label,
        band_min_tons,
        band_max_tons,
        rate_sdr_per_ton
    FROM toll_rate_bands
    ORDER BY id
    """,
    conn
)

print("Shape:", toll_bands_db.shape)
print(toll_bands_db.head())

Shape: (177, 8)
   id    vessel_category laden_ballast  band_order       band_label  \
0   1  Crude Oil Tankers         Laden           1  First 5000 Tons   
1   2  Crude Oil Tankers         Laden           2   Next 5000 Tons   
2   3  Crude Oil Tankers         Laden           3  Next 10000 Tons   
3   4  Crude Oil Tankers         Laden           4  Next 20000 Tons   
4   5  Crude Oil Tankers         Laden           5  Next 30000 Tons   

   band_min_tons  band_max_tons  rate_sdr_per_ton  
0            0.0         5000.0              9.60  
1         5000.0        10000.0              6.80  
2        10000.0        20000.0              5.14  
3        20000.0        40000.0              2.55  
4        40000.0        70000.0              2.20  


C:\Users\terbo\AppData\Local\Temp\ipykernel_22536\1145229132.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  toll_bands_db = pd.read_sql_query(


In [30]:
# Lookup table using the real PostgreSQL band IDs
def get_toll_band_db_id(vessel_category, condition, scnt_tons):
    
    # Special Floating Units has Laden rates only
    if vessel_category == "Special Floating Units":
        condition = "Laden"

    bands = toll_bands_db[
        (toll_bands_db["vessel_category"] == vessel_category) &
        (toll_bands_db["laden_ballast"] == condition) &
        (toll_bands_db["band_min_tons"] < scnt_tons) &
        (
            toll_bands_db["band_max_tons"].isna() |
            (toll_bands_db["band_max_tons"] >= scnt_tons)
        )
    ].sort_values("band_order")

    if bands.empty:
        return None

    return int(bands.iloc[0]["id"])

In [31]:
# Add the real PostgreSQL toll_rate_band_id
vessel_lookup = vessels_df.set_index("imo_number")[
    ["vessel_type", "condition"]
]

invoices_df["toll_rate_band_id"] = invoices_df.apply(
    lambda row: get_toll_band_db_id(
        vessel_lookup.loc[row["vessel_imo"], "vessel_type"],
        vessel_lookup.loc[row["vessel_imo"], "condition"],
        row["scnt_tons"]
    ),
    axis=1
)

print("Missing toll_rate_band_id:",
      invoices_df["toll_rate_band_id"].isnull().sum())

Missing toll_rate_band_id: 0


In [32]:
print(invoices_df.shape)
print("Missing vessel_id:", invoices_df["vessel_id"].isnull().sum())
print("Missing toll_rate_band_id:", invoices_df["toll_rate_band_id"].isnull().sum())

print(
    invoices_df[
        [
            "invoice_number",
            "vessel_id",
            "toll_rate_band_id",
            "invoice_date",
            "scnt_tons",
            "amount",
            "status"
        ]
    ].head()
)

(1000, 9)
Missing vessel_id: 0
Missing toll_rate_band_id: 0
  invoice_number  vessel_id  toll_rate_band_id invoice_date  scnt_tons  \
0    INV-0000001         17                 89   2025-11-24   62949.78   
1    INV-0000002         83                117   2025-05-21   51036.72   
2    INV-0000003         42                 48   2025-03-13   96933.52   
3    INV-0000004        370                 89   2025-10-07   47483.36   
4    INV-0000005        154                  7   2025-08-05  187150.54   

      amount  status  
0  300311.68    Paid  
1  237291.91    Paid  
2  440741.34  Issued  
3  242312.60  Issued  
4  469128.50    Paid  


In [33]:
 invoices_df = invoices_df[
    [
        "invoice_number",
        "vessel_id",
        "toll_rate_band_id",
        "invoice_date",
        "scnt_tons",
        "amount",
        "description",
        "status"
    ]
]

print("Shape:", invoices_df.shape)
print("Columns:", invoices_df.columns.tolist())

IndentationError: unexpected indent (408871482.py, line 1)